# LLM Exposure Index – Indexberechnung und Analysevorbereitung
**Masterarbeit | Kapitel 5.5–5.7**
Autor: Ayumi Nojima | April 2026

---
## Outputs dieses Notebooks

| Datei | Inhalt | Kapitel |
|-------|--------|---------|
| `analysis_prep/mu_weights.csv` | LLM-Gewichte μ_i | 5.5 |
| `analysis_prep/final_sample.csv` | Finales Analysesample | 5.7 |
| `analysis_prep/skill_vectors_standardized.csv` | Standardisierte Fähigkeitsvektoren | 5.7 |
| `output/dataset_H1.csv` | H1: Gruppenunterschiede (ANOVA) | 5.7 |
| `output/dataset_H2.csv` | H2: inkl. isco_4digit + delta_bfs für Option B | 5.7 |
| `output/dataset_H3.csv` | H3: Hauptanalyse 2022→2024 | 5.7 |
| `output/dataset_H3_panel.csv` | **NEU:** H3 Panel 2021–2024 (Long-Format) | 5.7 |
| `output/dataset_validation.csv` | Konvergente Validierung | 5.7 |

**Änderungen gegenüber Vorversion:**
- H2-Datensatz enthält neu `isco_4digit` und `delta_bfs` → ermöglicht Option B (RF → ΔBFS)
- H3-Panel-Datensatz (Long-Format, 3 Perioden) für Fixed-Effects-Analyse
- Vollständigkeitsprüfung um Panel-Outputs erweitert

**Voraussetzung:** Notebook 1 vollständig durchgelaufen.


## 0. Imports und Konfiguration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings("ignore")

# ── Repo-Root robust bestimmen ─────────────────────────────────────────────
_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if (_cwd / "..").resolve().joinpath("data").exists() else _cwd
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = Path.cwd()
print("Repo-Root:", REPO_ROOT)

# ── Pfade ──────────────────────────────────────────────────────────────────
PROCESSED = REPO_ROOT / "data" / "processed"
ANALYSIS  = PROCESSED / "analysis_prep"
OUTPUT    = REPO_ROOT / "data" / "output"
for p in [ANALYSIS, OUTPUT]:
    p.mkdir(parents=True, exist_ok=True)

# ── Parameter ──────────────────────────────────────────────────────────────
# HINWEIS: SKILL_OVERLAP_THRESHOLD wird nicht mehr als binärer Filter verwendet.
# s_ij wird als kontinuierliches Gewicht eingesetzt (vgl. Kap. 5.6).
# Grund: Binärer Filter bei 0.70 führte dazu, dass kein Beruf die Schwelle
# erreichte (max. s_ij = 0.638), da die Pearson-Korrelation zwischen dem
# 161-dimensionalen O*NET-Profil und dem 50-dimensionalen μ_i-Vektor
# strukturell begrenzt ist. Kontinuierliches Gewicht ist methodisch kohärenter.
MAIN_GROUPS     = [1, 2, 3]
BFS_BASE_YEAR   = 2022
BFS_TARGET_YEAR = 2024

# ── Daten aus Notebook 1 laden ─────────────────────────────────────────────
onet_pivot    = pd.read_csv(PROCESSED / "onet_pivot.csv")
train_mapping = pd.read_csv(PROCESSED / "train_mapping.csv")
ch_isco       = pd.read_csv(PROCESSED / "ch_isco_clean.csv")
bfs           = pd.read_csv(PROCESSED / "bfs_clean.csv")

print(f"onet_pivot:    {onet_pivot.shape[0]} Berufe × {onet_pivot.shape[1]-1} Dimensionen ✓")
print(f"train_mapping: {len(train_mapping)} Berufe ✓")
print(f"ch_isco:       {len(ch_isco)} Klassen ✓")
print(f"bfs:           {len(bfs)} Berufsgruppen ✓")
print("Konfiguration geladen ✓")


Repo-Root: /workspaces/Master_Thesis
onet_pivot:    894 Berufe × 161 Dimensionen ✓
train_mapping: 571 Berufe ✓
ch_isco:       271 Klassen ✓
bfs:           215 Berufsgruppen ✓
Konfiguration geladen ✓


---
## 5.5 Kalibrierung der LLM-Gewichte (μ_i)

μ_i-Gewichte basieren auf Eloundou et al. (2023), adjustiert für aktuelle LLM-Benchmarks  
(OpenAI et al., 2024; DeepSeek-AI et al., 2025). Rescaling auf [0.1, 0.9] verhindert  
Extremnullgewichtungen. **Schwellenwerte:**
- μ_i > 0.7 → Substitutionsanteil (E^sub_j)
- 0.3 ≤ μ_i ≤ 0.7 → Augmentationsanteil (E^aug_j)
- μ_i < 0.3 → nicht in Index aufgenommen

**Zirkularitätsprävention:** Gewichte werden extern kalibriert (nicht aus Validierungsdaten  
geschätzt), um eine closed-loop-Situation zu vermeiden (vgl. Kap. 4.5.2).


In [2]:
# μ_i-Gewichte (Eloundou et al., 2023, adjustiert für GPT-4o/Claude 4/DeepSeek V3)
# Präfix-Format muss mit element_name in onet_pivot übereinstimmen
mu_weights_raw = {
    # ── Skills ──────────────────────────────────────────────────────────────
    "Skills – Reading Comprehension":           0.85,
    "Skills – Writing":                         0.88,
    "Skills – Active Listening":                0.55,
    "Skills – Speaking":                        0.45,
    "Skills – Mathematics":                     0.80,
    "Skills – Science":                         0.55,
    "Skills – Critical Thinking":               0.82,
    "Skills – Active Learning":                 0.60,
    "Skills – Learning Strategies":             0.50,
    "Skills – Monitoring":                      0.45,
    "Skills – Social Perceptiveness":           0.30,
    "Skills – Coordination":                    0.35,
    "Skills – Persuasion":                      0.40,
    "Skills – Negotiation":                     0.38,
    "Skills – Instructing":                     0.42,
    "Skills – Service Orientation":             0.35,
    "Skills – Complex Problem Solving":         0.80,
    "Skills – Operations Analysis":             0.70,
    "Skills – Technology Design":               0.55,
    "Skills – Equipment Selection":             0.20,
    "Skills – Installation":                    0.10,
    "Skills – Programming":                     0.90,
    "Skills – Quality Control Analysis":        0.50,
    "Skills – Operations Monitoring":           0.35,
    "Skills – Operation and Control":           0.15,
    "Skills – Equipment Maintenance":           0.10,
    "Skills – Troubleshooting":                 0.45,
    "Skills – Repairing":                       0.10,
    "Skills – Systems Analysis":                0.72,
    "Skills – Systems Evaluation":              0.68,
    "Skills – Time Management":                 0.45,
    "Skills – Management of Financial Resources": 0.55,
    "Skills – Management of Material Resources":  0.25,
    "Skills – Management of Personnel Resources": 0.40,
    # ── Work Activities ──────────────────────────────────────────────────────
    "Work Activities – Getting Information":                   0.80,
    "Work Activities – Processing Information":                0.85,
    "Work Activities – Analyzing Data or Information":         0.82,
    "Work Activities – Making Decisions and Solving Problems": 0.75,
    "Work Activities – Thinking Creatively":                   0.70,
    "Work Activities – Updating and Using Relevant Knowledge": 0.72,
    "Work Activities – Communicating with Supervisors, Peers, or Subordinates": 0.40,
    "Work Activities – Communicating with People Outside the Organization":     0.38,
    "Work Activities – Establishing and Maintaining Interpersonal Relationships": 0.25,
    "Work Activities – Developing Objectives and Strategies":  0.65,
    "Work Activities – Organizing, Planning, and Prioritizing Work": 0.55,
    "Work Activities – Performing Administrative Activities":  0.75,
    "Work Activities – Documenting/Recording Information":     0.80,
    "Work Activities – Interpreting the Meaning of Information for Others": 0.72,
    "Work Activities – Provide Consultation and Advice to Others": 0.60,
    # ── Knowledge ────────────────────────────────────────────────────────────
    "Knowledge – English Language":              0.85,
    "Knowledge – Mathematics":                   0.80,
    "Knowledge – Computers and Electronics":     0.82,
    "Knowledge – Administration and Management": 0.60,
    "Knowledge – Economics and Accounting":      0.70,
    "Knowledge – Law and Government":            0.65,
    "Knowledge – Medicine and Dentistry":        0.45,
    "Knowledge – Education and Training":        0.55,
}

# Rescaling auf [0.1, 0.9]
mu_min = min(mu_weights_raw.values())
mu_max = max(mu_weights_raw.values())
mu_weights = {
    k: round(0.1 + (v - mu_min) / (mu_max - mu_min) * 0.8, 4)
    for k, v in mu_weights_raw.items()
}

mu_df = pd.DataFrame([
    {
        "element_name":  k,
        "mu_i":          v,
        "exposure_type": "substitution" if v > 0.7 else ("augmentation" if v >= 0.3 else "negligible")
    }
    for k, v in mu_weights.items()
])

print(f"μ_i-Gewichte definiert: {len(mu_df)} Dimensionen")
print(mu_df["exposure_type"].value_counts().to_string())

mu_df.to_csv(ANALYSIS / "mu_weights.csv", index=False)
print("\nGespeichert → data/processed/analysis_prep/mu_weights.csv ✓")


μ_i-Gewichte definiert: 57 Dimensionen
exposure_type
augmentation    32
substitution    18
negligible       7

Gespeichert → data/processed/analysis_prep/mu_weights.csv ✓


---
## 5.6 Berechnung des Exposure-Index E_j

$$E_j = \sum_{i=1}^{n} \mu_i \cdot w_{ij} \cdot s_{ij}$$

- $w_{ij}$ = Min-Max-normalisierter Importance-Wert (O\*NET)
- $\mu_i$ = LLM-Gewicht der Fähigkeitsdimension i
- $s_{ij}$ = Skill-Overlap-Koeffizient als **kontinuierliches Gewicht** [0,1]

**Methodische Anpassung gegenüber ursprünglichem Design:**  
Der binäre Filter (s_ij ≥ 0.70) wurde durch ein kontinuierliches Gewicht ersetzt.  
Grund: Der maximale s_ij-Wert im Datensatz beträgt 0.638 — kein Beruf hätte  
die binäre Schwelle erreicht, da die Pearson-Korrelation zwischen dem  
161-dimensionalen O\*NET-Profil und dem 50-dimensionalen μ_i-Vektor  
strukturell begrenzt ist. Das kontinuierliche Gewicht ist methodisch kohärenter:  
ein höherer s_ij skaliert den Indexwert stärker, ohne Berufe binär auszuschliessen.


In [3]:
# ── Gemeinsame Dimensionen ────────────────────────────────────────────────
mu_active   = mu_df[mu_df["exposure_type"] != "negligible"].set_index("element_name")["mu_i"]
common_dims = [c for c in onet_pivot.columns if c in mu_active.index]
print(f"Gemeinsame Fähigkeitsdimensionen für Indexberechnung: {len(common_dims)}")

onet_matrix = onet_pivot.set_index("soc_code")[common_dims].fillna(0)
mu_vector   = mu_active[common_dims].values

# LLM-Capability-Referenzprofil (normierter μ-Vektor)
llm_profile = mu_vector / (np.linalg.norm(mu_vector) + 1e-9)

# ── Skill-Overlap-Koeffizient s_ij (Pearson-Korrelation) ──────────────────
def skill_overlap(row_vals, reference):
    if np.std(row_vals) < 1e-9:
        return 0.0
    r, _ = pearsonr(row_vals, reference)
    return max(r, 0.0)

sij_values = onet_matrix.apply(lambda row: skill_overlap(row.values, llm_profile), axis=1)

# Kontinuierliches Gewicht statt binärem Filter
# Negative Korrelationen (anti-korreliert mit LLM-Profil) → 0
sij_continuous = sij_values.clip(lower=0)

print(f"\ns_ij Verteilung (kontinuierlich):")
print(f"  Mean={sij_continuous.mean():.3f} | Max={sij_continuous.max():.3f} | "
      f"Min={sij_continuous.min():.3f}")
print(f"  Berufe mit s_ij > 0: {(sij_continuous > 0).sum()} von {len(sij_continuous)}")
print(f"  Berufe mit s_ij > 0.3: {(sij_continuous > 0.3).sum()}")
print(f"  Berufe mit s_ij > 0.5: {(sij_continuous > 0.5).sum()}")

# ── E_j berechnen ──────────────────────────────────────────────────────────
weighted_sum = onet_matrix.dot(mu_vector)
E_j_raw      = weighted_sum * sij_continuous

# Subindizes (NaN-sicher via np.where)
mu_vals = mu_active[common_dims].values
mu_sub  = np.where(mu_vals > 0.7,                             mu_vals, 0)
mu_aug  = np.where((mu_vals >= 0.3) & (mu_vals <= 0.7),       mu_vals, 0)

E_sub_raw = onet_matrix.dot(mu_sub) * sij_continuous
E_aug_raw = onet_matrix.dot(mu_aug) * sij_continuous

# Min-Max-Normierung auf [0, 1]
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

index_df = pd.DataFrame({
    "soc_code": onet_matrix.index,
    "s_ij":     sij_continuous.values,
    "E_j":      minmax(E_j_raw).values,
    "E_sub_j":  minmax(E_sub_raw).values,
    "E_aug_j":  minmax(E_aug_raw).values,
})

print(f"\nIndex berechnet: {len(index_df)} Berufe")
print(f"E_j   – Mean: {index_df['E_j'].mean():.3f} | Std: {index_df['E_j'].std():.3f}")
print(f"E^sub – Mean: {index_df['E_sub_j'].mean():.3f} | Std: {index_df['E_sub_j'].std():.3f}")
print(f"E^aug – Mean: {index_df['E_aug_j'].mean():.3f} | Std: {index_df['E_aug_j'].std():.3f}")


Gemeinsame Fähigkeitsdimensionen für Indexberechnung: 49



s_ij Verteilung (kontinuierlich):
  Mean=0.142 | Max=0.638 | Min=0.000
  Berufe mit s_ij > 0: 618 von 894
  Berufe mit s_ij > 0.3: 164
  Berufe mit s_ij > 0.5: 23

Index berechnet: 894 Berufe
E_j   – Mean: 0.197 | Std: 0.231
E^sub – Mean: 0.178 | Std: 0.213
E^aug – Mean: 0.223 | Std: 0.257


### Monte-Carlo-Konfidenzintervall (Mapping-Konfidenz)

In [4]:
np.random.seed(42)
N_ITER = 1000
mc_results = []

for _ in range(N_ITER):
    mu_noisy = mu_vector * (1 + np.random.normal(0, 0.05, size=len(mu_vector)))
    mu_noisy = np.clip(mu_noisy, 0.1, 0.9)
    # Kontinuierliches s_ij auch in Monte-Carlo
    E_mc = minmax(onet_matrix.dot(mu_noisy) * sij_continuous)
    mc_results.append(E_mc.values)

mc_array = np.array(mc_results)
ci_half  = np.percentile(mc_array, 97.5, axis=0) - np.percentile(mc_array, 2.5, axis=0)
mean_ci  = ci_half.mean() / 2 * 100

print(f"Monte-Carlo ({N_ITER} Iterationen):")
print(f"  Mittleres 95%-KI: ±{mean_ci:.1f}%  (Zielwert: ±8%)")
print(f"  Status: {'✓ Zielwert erreicht' if mean_ci <= 8 else '⚠ Zielwert überschritten – vgl. Kap. 6.5'}")


Monte-Carlo (1000 Iterationen):
  Mittleres 95%-KI: ±0.1%  (Zielwert: ±8%)
  Status: ✓ Zielwert erreicht


---
## 5.7 Vorbereitung der Analysedatensätze

Datensätze für die drei Analysestufen aus Kapitel 4.8.


In [5]:
# ── Finales Sample zusammenführen ─────────────────────────────────────────
final = (
    index_df
    .merge(train_mapping[["soc_code", "isco_4digit", "mapping_stage"]], on="soc_code", how="inner")
    .merge(ch_isco[["ch_isco_4digit", "ch_isco_label", "main_group"]],
           left_on="isco_4digit", right_on="ch_isco_4digit", how="left")
    .merge(bfs[["ch_isco_4digit", f"employed_{BFS_BASE_YEAR}", f"employed_{BFS_TARGET_YEAR}",
                "delta_bfs", "is_outlier"]],
           on="ch_isco_4digit", how="left")
)

print(f"Finales Analysesample: {len(final)} Berufe")
print(f"  Hauptgruppe 1 (Führungskräfte):  {(final['main_group']==1).sum()}")
print(f"  Hauptgruppe 2 (Akademisch):      {(final['main_group']==2).sum()}")
print(f"  Hauptgruppe 3 (Techniker):       {(final['main_group']==3).sum()}")
print(f"  Ohne BFS-Match (NaN delta_bfs):  {final['delta_bfs'].isna().sum()}")
print()

# Schnellcheck: E_j muss Varianz haben
assert final['E_j'].std() > 0, "FEHLER: E_j hat keine Varianz – Index-Berechnung prüfen!"
print(f"E_j Check: Mean={final['E_j'].mean():.3f} | Std={final['E_j'].std():.3f} ✓")


Finales Analysesample: 571 Berufe
  Hauptgruppe 1 (Führungskräfte):  62
  Hauptgruppe 2 (Akademisch):      296
  Hauptgruppe 3 (Techniker):       213
  Ohne BFS-Match (NaN delta_bfs):  21

E_j Check: Mean=0.230 | Std=0.239 ✓


In [6]:
# ── H1: Gruppenunterschiede nach CH-ISCO-Hauptgruppe ─────────────────────
h1 = final[["soc_code", "isco_4digit", "main_group", "E_j", "E_sub_j", "E_aug_j"]].copy()
h1.to_csv(OUTPUT / "dataset_H1.csv", index=False)
print(f"H1-Datensatz: {len(h1)} Berufe → data/output/dataset_H1.csv ✓")

# ── H2: Fähigkeitsdimensionen als Prädiktoren ────────────────────────────
# HINWEIS: H2 wird in 06_2_hypothesen.ipynb in zwei Optionen aufgeteilt:
#   Option A: μ_i-Gewichte direkt visualisieren (keine Zirkularität)
#   Option B: RF Fähigkeiten → ΔBFS_j auf ISCO-Ebene (NEU, zirkulationsfrei)
#
# Für Option B: isco_4digit und delta_bfs als zusätzliche Spalten mitliefern
# damit in 06_2 direkt auf ISCO-Ebene aggregiert werden kann.
h2 = (
    final[["soc_code", "isco_4digit", "E_sub_j", "delta_bfs", "is_outlier"]]
    .merge(onet_pivot, on="soc_code", how="left")
)
h2.to_csv(OUTPUT / "dataset_H2.csv", index=False)
print(f"H2-Datensatz: {len(h2)} Berufe × {h2.shape[1]-5} Prädiktoren → data/output/dataset_H2.csv ✓")
print(f"  Enthält: isco_4digit + delta_bfs für Option B (RF → ΔBFS_j) ✓")

# ── H3: Hauptanalyse 2022→2024 ───────────────────────────────────────────
h3 = final[["soc_code", "isco_4digit", "main_group", "E_j", "delta_bfs", "is_outlier",
             f"employed_{BFS_BASE_YEAR}", f"employed_{BFS_TARGET_YEAR}"]].copy()
h3["qualification_level"] = h3["main_group"].map({1: 3, 2: 4, 3: 3})
h3["sector"] = "Sonstiges"
h3.to_csv(OUTPUT / "dataset_H3.csv", index=False)
print(f"H3-Datensatz: {len(h3)} Berufe → data/output/dataset_H3.csv ✓")

# ── H3 Panel: 2021–2024 (Prio 2: Fixed-Effects Panel-Analyse) ────────────
# Panel-BFS laden (erstellt in Notebook 1)
try:
    bfs_panel = pd.read_csv(PROCESSED / "bfs_panel_2021_2024.csv",
                             dtype={"ch_isco_4digit": str})

    # E_j je ISCO-Code aggregieren (Ø über O*NET-Berufe)
    ej_isco = (
        final.groupby("isco_4digit")
        .agg(
            E_j_mean      = ("E_j",       "mean"),
            E_sub_mean    = ("E_sub_j",   "mean"),
            E_aug_mean    = ("E_aug_j",   "mean"),
            main_group    = ("main_group","first"),
            n_onet_berufe = ("soc_code",  "count"),
        )
        .reset_index()
    )
    ej_isco["isco_4digit"] = ej_isco["isco_4digit"].astype(str)
    bfs_panel["ch_isco_4digit"] = bfs_panel["ch_isco_4digit"].astype(str)

    # Panel-Masterdatensatz: E_j + 3 Jahresschritte
    h3_panel = bfs_panel.merge(
        ej_isco.rename(columns={"isco_4digit": "ch_isco_4digit"}),
        on="ch_isco_4digit", how="inner"
    )

    # Post-KI Dummy
    # 2021→2022 = Vor-KI (post=0); 2022→2023, 2023→2024 = Nach-KI (post=1)
    h3_panel["post_ki_2022_2023"] = 1  # 2022→2023
    h3_panel["post_ki_2023_2024"] = 1  # 2023→2024
    # (2021→2022 bleibt die Referenzperiode ohne explizite Spalte)

    # Long-Format für Fixed-Effects
    h3_panel_long = pd.concat([
        h3_panel[["ch_isco_4digit", "E_j_mean", "E_sub_mean", "E_aug_mean",
                   "main_group", "n_onet_berufe"]]
        .assign(periode="2021_2022",
                delta_bfs=h3_panel["delta_2021_2022"],
                post_ki=0),
        h3_panel[["ch_isco_4digit", "E_j_mean", "E_sub_mean", "E_aug_mean",
                   "main_group", "n_onet_berufe"]]
        .assign(periode="2022_2023",
                delta_bfs=h3_panel["delta_2022_2023"],
                post_ki=1),
        h3_panel[["ch_isco_4digit", "E_j_mean", "E_sub_mean", "E_aug_mean",
                   "main_group", "n_onet_berufe"]]
        .assign(periode="2023_2024",
                delta_bfs=h3_panel["delta_2023_2024"],
                post_ki=1),
    ], ignore_index=True)

    h3_panel_long.to_csv(OUTPUT / "dataset_H3_panel.csv", index=False)
    print(f"H3-Panel-Datensatz: {len(h3_panel)} ISCO-Codes × 3 Perioden = "
          f"{len(h3_panel_long)} Beobachtungen → data/output/dataset_H3_panel.csv ✓")
    print(f"  Vor-KI  (post=0): {(h3_panel_long['post_ki']==0).sum()} Beobachtungen")
    print(f"  Nach-KI (post=1): {(h3_panel_long['post_ki']==1).sum()} Beobachtungen")

except FileNotFoundError:
    print("⚠ bfs_panel_2021_2024.csv nicht gefunden – Notebook 1 zuerst ausführen")
    print("  H3-Panel wird übersprungen")

# ── Konvergente Validierung ────────────────────────────────────────────────
validation = final[["soc_code", "isco_4digit", "E_j", "E_sub_j", "E_aug_j"]].copy()
validation["kuprecht_2025_score"] = np.nan
validation.to_csv(OUTPUT / "dataset_validation.csv", index=False)
print(f"Validierungs-Datensatz → data/output/dataset_validation.csv ✓")

# ── Standardisierte Fähigkeitsvektoren ────────────────────────────────────
skill_std = onet_pivot.set_index("soc_code")[common_dims].copy()
skill_std = (skill_std - skill_std.mean()) / (skill_std.std() + 1e-9)
skill_std.to_csv(ANALYSIS / "skill_vectors_standardized.csv")
print(f"Standardisierte Fähigkeitsvektoren → data/processed/analysis_prep/ ✓")


H1-Datensatz: 571 Berufe → data/output/dataset_H1.csv ✓
H2-Datensatz: 571 Berufe × 161 Prädiktoren → data/output/dataset_H2.csv ✓
  Enthält: isco_4digit + delta_bfs für Option B (RF → ΔBFS_j) ✓
H3-Datensatz: 571 Berufe → data/output/dataset_H3.csv ✓
H3-Panel-Datensatz: 155 ISCO-Codes × 3 Perioden = 465 Beobachtungen → data/output/dataset_H3_panel.csv ✓
  Vor-KI  (post=0): 155 Beobachtungen
  Nach-KI (post=1): 310 Beobachtungen
Validierungs-Datensatz → data/output/dataset_validation.csv ✓
Standardisierte Fähigkeitsvektoren → data/processed/analysis_prep/ ✓


---
## Finales Sample speichern

`final_sample.csv` ist der zentrale Masterdatensatz für alle Analyse-Notebooks (06_1–06_4).


In [7]:
# ── Finales Sample als CSV speichern ──────────────────────────────────────
final.to_csv(ANALYSIS / "final_sample.csv", index=False)
print(f"Finales Sample gespeichert: {len(final)} Berufe")
print(f"  Spalten ({len(final.columns)}): {final.columns.tolist()}")
print(f"  Hauptgruppe 1: {(final['main_group']==1).sum()}")
print(f"  Hauptgruppe 2: {(final['main_group']==2).sum()}")
print(f"  Hauptgruppe 3: {(final['main_group']==3).sum()}")
print(f"  Mit ΔBFS_j:    {final['delta_bfs'].notna().sum()}")
print(f"  E_j > 0:       {(final['E_j'] > 0).sum()}")
print()
print("Gespeichert → data/processed/analysis_prep/final_sample.csv ✓")


Finales Sample gespeichert: 571 Berufe
  Spalten (14): ['soc_code', 's_ij', 'E_j', 'E_sub_j', 'E_aug_j', 'isco_4digit', 'mapping_stage', 'ch_isco_4digit', 'ch_isco_label', 'main_group', 'employed_2022', 'employed_2024', 'delta_bfs', 'is_outlier']
  Hauptgruppe 1: 62
  Hauptgruppe 2: 296
  Hauptgruppe 3: 213
  Mit ΔBFS_j:    550
  E_j > 0:       415

Gespeichert → data/processed/analysis_prep/final_sample.csv ✓


---
## Vollständigkeitsprüfung – alle Outputs vorhanden?


In [8]:
# ── Vollständigkeitsprüfung ───────────────────────────────────────────────
expected_files = {
    "Notebook 06_1 (EDA)": [
        ANALYSIS / "final_sample.csv",
        ANALYSIS / "skill_vectors_standardized.csv",
        ANALYSIS / "mu_weights.csv",
    ],
    "Notebook 06_2 (Hypothesen)": [
        ANALYSIS / "final_sample.csv",
        OUTPUT   / "dataset_H1.csv",
        OUTPUT   / "dataset_H2.csv",     # enthält jetzt isco_4digit + delta_bfs für Option B
        OUTPUT   / "dataset_H3.csv",
        OUTPUT   / "dataset_H3_panel.csv", # NEU: Panel 2021–2024
        PROCESSED / "onet_pivot.csv",
    ],
    "Notebook 06_3 (Validierung)": [
        ANALYSIS / "final_sample.csv",
        OUTPUT   / "dataset_validation.csv",
    ],
    "Notebook 06_4 (Erweiterungen)": [
        ANALYSIS / "final_sample.csv",
        REPO_ROOT / "data" / "raw" / "CH_ISCO19.xlsx",
    ],
    "Notebook 06_5 (Panel-Analyse) [NEU]": [
        OUTPUT   / "dataset_H3_panel.csv",
        PROCESSED / "bfs_panel_2021_2024.csv",
    ],
}

all_ok = True
for notebook, files in expected_files.items():
    print(f"\n{notebook}:")
    for f in files:
        exists = f.exists()
        status = "✓" if exists else "✗ FEHLT"
        size   = f"{f.stat().st_size / 1024:.1f} KB" if exists else ""
        print(f"  {status}  {f.name:<52} {size}")
        if not exists:
            all_ok = False

print()
if all_ok:
    print("=" * 60)
    print("  Alle Dateien vorhanden – bereit für Analyse-Notebooks")
    print("=" * 60)
else:
    print("⚠ Fehlende Dateien – Notebooks erneut ausführen")



Notebook 06_1 (EDA):
  ✓  final_sample.csv                                     105.0 KB
  ✓  skill_vectors_standardized.csv                       849.5 KB
  ✓  mu_weights.csv                                       3.1 KB

Notebook 06_2 (Hypothesen):
  ✓  final_sample.csv                                     105.0 KB
  ✓  dataset_H1.csv                                       35.6 KB
  ✓  dataset_H2.csv                                       1618.2 KB
  ✓  dataset_H3.csv                                       57.7 KB
  ✓  dataset_H3_panel.csv                                 42.0 KB
  ✓  onet_pivot.csv                                       2566.9 KB

Notebook 06_3 (Validierung):
  ✓  final_sample.csv                                     105.0 KB
  ✓  dataset_validation.csv                               35.1 KB

Notebook 06_4 (Erweiterungen):
  ✓  final_sample.csv                                     105.0 KB
  ✓  CH_ISCO19.xlsx                                       2841.4 KB

Notebook 06_5 (Pan